In [1]:
!pip install -q --upgrade streamlit transformers torch accelerate bitsandbytes openai-whisper pyngrok

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 803.2/803.2 kB 20.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 96.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 899.7/899.7 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 795.5 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 103.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 10.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 51.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.4 MB/s et

In [2]:
%%writefile app.py
import streamlit as st
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration, AutoTokenizer, AutoModelForCausalLM, pipeline
import whisper
import torch
import datetime
import smtplib
from email.mime.text import MIMEText
from email.mime.multipart import MIMEMultipart

# --- 1. CONFIGURATION ---
st.set_page_config(page_title="DermoAI Clinic", page_icon="🏥", layout="centered")

# --- 2. SESSION STATE (The "Memory") ---
if "history" not in st.session_state:
    st.session_state.history = []  # Stores the chat
if "stage" not in st.session_state:
    st.session_state.stage = "greeting"  # greeting -> triage -> consultation
if "patient_data" not in st.session_state:
    st.session_state.patient_data = {}
if "booking_status" not in st.session_state:
    st.session_state.booking_status = "pending" # pending -> booked -> not_needed

device = "cuda" if torch.cuda.is_available() else "cpu"

# --- 3. LOAD MODELS ---
@st.cache_resource
def load_models():
    # Vision
    blip_proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
    blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
    # Audio
    whisper_model = whisper.load_model("base")
    # Brain (Qwen2.5 - Best Small Medical Model)
    tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
    model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct", device_map="auto", torch_dtype=torch.float16)
    brain = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=400)
    return blip_proc, blip_model, whisper_model, brain

blip_proc, blip_model, whisper_model, brain = load_models()

# --- 4. REAL EMAIL FUNCTION ---
def send_email_appointment(user_email, date, time, diagnosis):
    # SETTINGS: To make this work, you need a Gmail App Password
    # If you don't have one, this function will just simulate success.
    SENDER_EMAIL = "dermoai.clinic@gmail.com" # Replace with your email
    APP_PASSWORD = "xxxx xxxx xxxx xxxx"      # Replace with your App Password

    msg = MIMEMultipart()
    msg['From'] = SENDER_EMAIL
    msg['To'] = user_email
    msg['Subject'] = "Appointment Confirmation - DermoAI Clinic"

    body = f"""
    Dear Patient,

    Your appointment with Dr. DermoAI is confirmed.

     Date: {date}
     Time: {time}

    --- MEDICAL RECORD ---
    Diagnosis: {diagnosis}

    Please arrive 10 minutes early.
    """
    msg.attach(MIMEText(body, 'plain'))

    try:
        # Connect to Gmail Server
        server = smtplib.SMTP('smtp.gmail.com', 587)
        server.starttls()
        server.login(SENDER_EMAIL, APP_PASSWORD)
        text = msg.as_string()
        server.sendmail(SENDER_EMAIL, user_email, text)
        server.quit()
        return True, "Email sent successfully!"
    except Exception as e:
        # Fallback simulation if no password is set
        return False, f"Simulated Email: {user_email} (Configure SMTP for real sending)"

# --- 5. CHAT LOGIC ---

st.title(" DermoAI Virtual Clinic")

# Display the conversation
for msg in st.session_state.history:
    with st.chat_message(msg["role"]):
        st.markdown(msg["content"])

# --- STAGE 1: GREETING ---
if st.session_state.stage == "greeting":
    if not st.session_state.history:
        greeting = "Hello. I am Dr. AI. I can help examine skin issues. Please briefly describe what you are feeling."
        st.session_state.history.append({"role": "assistant", "content": greeting})
        st.rerun()

    user_text = st.chat_input("Describe your problem...")
    if user_text:
        st.session_state.history.append({"role": "user", "content": user_text})
        st.session_state.patient_data["complaint"] = user_text

        # Transition to Triage
        st.session_state.history.append({"role": "assistant", "content": "I understand. To give you a proper diagnosis, I need to examine the area. Please **upload a photo** and **record a voice note** describing the symptoms in detail."})
        st.session_state.stage = "triage"
        st.rerun()

# --- STAGE 2: TRIAGE (Uploads) ---
if st.session_state.stage == "triage":
    with st.container(border=True):
        st.info(" Patient Examination")
        col1, col2 = st.columns(2)
        with col1:
            uploaded_file = st.file_uploader("Attach Photo", type=["jpg", "png"])
        with col2:
            audio_value = st.audio_input("Record Voice")

        if st.button("Analyze Condition"):
            if uploaded_file and audio_value:
                with st.spinner("Dr. AI is examining the patient..."):
                    # 1. Vision Analysis
                    image = Image.open(uploaded_file).convert('RGB')
                    inputs = blip_proc(image, return_tensors="pt").to(device)
                    out = blip_model.generate(**inputs)
                    visual_findings = blip_proc.decode(out[0], skip_special_tokens=True)

                    # 2. Audio Analysis
                    with open("temp.wav", "wb") as f:
                        f.write(audio_value.read())
                    audio_text = whisper_model.transcribe("temp.wav")["text"]

                    # 3. Doctor's Brain (LLM)
                    prompt = f"""
                    Role: You are a professional, empathetic Dermatologist.
                    Patient Complaint: {st.session_state.patient_data.get('complaint')}
                    Visual Observation: {visual_findings}
                    Audio Symptoms: {audio_text}

                    Task:
                    1. Diagnose the condition based on the inputs.
                    2. DECIDE URGENCY:
                       - If symptoms include bleeding, growing fast, pain, black color, or irregular shapes -> MARK AS URGENT.
                       - Otherwise -> MARK AS ROUTINE.

                    Output Format:
                    Start with "DIAGNOSIS: [Urgent/Routine]".
                    Then write a paragraph explaining the condition to the patient.
                    If Routine: Suggest home care remedies.
                    If Urgent: Explain why they need a doctor immediately.
                    """

                    messages = [{"role": "user", "content": prompt}]
                    output = brain(messages)
                    full_response = output[0]['generated_text'][-1]['content']

                    # Parse Urgency
                    is_urgent = "URGENT" in full_response.upper()
                    st.session_state.patient_data["urgency"] = "Urgent" if is_urgent else "Routine"
                    st.session_state.patient_data["diagnosis_text"] = full_response

                    # Add Doctor's Response to Chat
                    st.session_state.history.append({"role": "assistant", "content": full_response})

                    # Set Next Stage
                    st.session_state.stage = "consultation"
                    st.rerun()

# --- STAGE 3: CONSULTATION (Split Path) ---
if st.session_state.stage == "consultation":

    # PATH A: URGENT CASE (Must Book)
    if st.session_state.patient_data.get("urgency") == "Urgent" and st.session_state.booking_status == "pending":
        st.error(" **This condition requires medical attention.**")
        with st.expander(" Book Priority Appointment", expanded=True):
            d = st.date_input("Date")
            t = st.time_input("Time")
            email = st.text_input("Email Address")

            if st.button("Confirm Appointment"):
                if email:
                    success, msg = send_email_appointment(email, d, t, st.session_state.patient_data["diagnosis_text"][:200])
                    st.session_state.booking_status = "booked"

                    confirmation_msg = f" **Appointment Confirmed** for {d} at {t}. I have sent the details to {email}.\n\nWhile you wait for your appointment, do you have any specific questions about pain management?"
                    st.session_state.history.append({"role": "assistant", "content": confirmation_msg})
                    st.rerun()
                else:
                    st.warning("Please enter an email.")

    # PATH B: ROUTINE CASE (Home Care - No Forced Booking)
    elif st.session_state.patient_data.get("urgency") == "Routine" and st.session_state.booking_status == "pending":
        # We auto-mark booking as "not needed" so they can just chat
        if st.session_state.booking_status != "not_needed":
            st.session_state.booking_status = "not_needed"
            follow_up = "Since this looks benign, you can treat it at home. Do you have questions about which creams or remedies to use?"
            st.session_state.history.append({"role": "assistant", "content": follow_up})
            st.rerun()

    # --- CONTINUOUS CHAT LOOP (For everyone) ---
    # This input is ALWAYS active after diagnosis, allowing endless conversation
    user_followup = st.chat_input("Ask Dr. AI a question...")

    if user_followup:
        st.session_state.history.append({"role": "user", "content": user_followup})

        # The AI answers contextually
        with st.spinner("Dr. AI is typing..."):
            context_prompt = f"""
            You are Dr. AI.
            Patient History: {st.session_state.patient_data.get('diagnosis_text')}
            Current Status: {st.session_state.patient_data.get('urgency')}
            User Question: {user_followup}

            Answer the question briefly and professionally.
            """
            messages = [{"role": "user", "content": context_prompt}]
            output = brain(messages)
            answer = output[0]['generated_text'][-1]['content']

            st.session_state.history.append({"role": "assistant", "content": answer})
            st.rerun()

Writing app.py


In [3]:
from pyngrok import ngrok

# PASTE YOUR TOKEN HERE
ngrok.set_auth_token("37ycCRYOUkRO4yDgNWZvsFYDuXp_22htCxi7GmnvQZjEiehb9")

ngrok.kill()
!streamlit run app.py --server.enableCORS=false --server.enableXsrfProtection=false --server.address=0.0.0.0 &>/content/logs.txt &

try:
    public_url = ngrok.connect(8501).public_url
    print(f" CLICK TO ENTER CLINIC: {public_url}")
except Exception as e:
    print(f"Error: {e}")

 CLICK TO ENTER CLINIC: https://unvascular-sammy-damply.ngrok-free.dev
